In [ ]:
import numpy as np
import pandas as pd
# from nltk import bigrams,trigrams
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,classification_report
from textblob import TextBlob
from sklearn.decomposition import TruncatedSVD
from nltk.corpus import stopwords
import nltk
from nltk.stem import PorterStemmer
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
import eli5
from eli5.sklearn import PermutationImportance
from sklearn.svm import SVC

In [ ]:
from tensorflow.keras import datasets # https://www.tensorflow.org/api_docs/python/tf/keras/datasets
from tensorflow.keras import layers # https://www.tensorflow.org/api_docs/python/tf/keras/layers
from tensorflow.keras import activations # https://www.tensorflow.org/api_docs/python/tf/keras/activations
from tensorflow.keras import initializers # https://www.tensorflow.org/api_docs/python/tf/keras/initializers
from tensorflow.keras import losses # https://www.tensorflow.org/api_docs/python/tf/keras/losses
from tensorflow.keras import metrics # https://www.tensorflow.org/api_docs/python/tf/keras/metrics
from tensorflow.keras import optimizers # https://www.tensorflow.org/api_docs/python/tf/keras/optimizers
from tensorflow.keras import regularizers # https://www.tensorflow.org/api_docs/python/tf/keras/regularizers
from tensorflow.keras.optimizers import schedules # https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/schedules
from tensorflow.keras import callbacks # https://www.tensorflow.org/api_docs/python/tf/keras/callbacks
from tensorflow.keras import utils # https://www.tensorflow.org/api_docs/python/tf/keras/utils
from tensorflow.keras import models # https://www.tensorflow.org/api_docs/python/tf/keras/models
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
#reading all the 17 features
all_features = pd.read_csv("featuresAll.csv")
labels = pd.read_csv("labels.csv")
all_features.head()

In [ ]:
all_features['labels'] = labels

In [ ]:
only_one = all_features.loc[all_features['labels'] == 1]
only_zero = all_features.loc[all_features['labels'] == 0]
train_one = only_one.iloc[:29508]
test_one = only_one.iloc[29508:]
train_zero = only_zero.iloc[:257733]
test_zero = only_zero.iloc[257733:]
train = train_one.append(train_zero, ignore_index = True)
test = test_one.append(test_zero, ignore_index = True)
train = train.sample(frac=1).reset_index(drop=True)
X_train = train.iloc[:,:17]
Y_train = train.iloc[:,17]
X_test = test.iloc[:,:17]
Y_test = test.iloc[:,17]
X_train = X_train.to_numpy()
Y_train = Y_train.to_numpy()
X_test = X_test.to_numpy()
Y_test = Y_test.to_numpy()
# np.random.shuffle(train)
# np.random.shuffle(test)

In [ ]:
# all_features = all_features.to_numpy()
# print(all_features.shape)

In [ ]:
# #reading the reviews
# file1 = open("YelpNYC/reviewContent")
# all_stop_words = stopwords.words('english')
# data1 = []
# for s in file1:
#     row = s[17:]
#     row = re.sub('[^a-zA-Z0-9 \n\..]', '', row)
#     row = ' '.join([word for word in row.split() if word not in all_stop_words])
#     data1.append(row)
# file1.close()
# print(len(data1))

In [ ]:
# #reading all the labels
# labels = pd.read_csv("labels.csv")
# Y = labels.to_numpy()
# Y.shape

## TF-IDF Vectorization

In [ ]:
# #using tfidf vectorization to convert the reviews into features
# tfidf_vector = TfidfVectorizer(stop_words='english')
# X_tfidf = tfidf_vector.fit_transform(data1)

In [ ]:
# #reducing the number of features
# svd = TruncatedSVD(n_components = 100)
# X_tfidf = svd.fit_transform(X_tfidf)

In [ ]:
# #putting all the features together
# # X1 = np.hstack((X_tfidf,all_features))
# X1 = all_features
# X1.shape

In [ ]:
model = tf.keras.Sequential([

    
    layers.Flatten(input_dim = len(X_train[0,:])), 
    
    layers.Dense(units=64, activation=None), 
    layers.BatchNormalization(),
    layers.Activation(activations.relu),
    
    layers.Dense(units=128, activation=None), 
    layers.BatchNormalization(),
    layers.Activation(activations.relu),
    
    layers.Dense(units=128, activation=None), 
    layers.BatchNormalization(),
    layers.Activation(activations.relu),
    
    layers.Dense(units=32, activation=None), 
    layers.BatchNormalization(),
    layers.Activation(activations.relu),
    
    layers.Dense(units=1, activation=activations.sigmoid) 
])

model.summary()

model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001), 
    loss=losses.BinaryCrossentropy(), 
    metrics=['accuracy']
) 

In [ ]:
model.compile(loss='binary_crossentropy',optimizer='adam',metrics='accuracy')

In [ ]:
model.fit(x=X_train, y=Y_train, epochs=10,batch_size=128, verbose=1,class_weight={0:1,1:7})

In [ ]:
predictions = np.round(model.predict(X_test))


print(Y_test.shape)
print(classification_report(Y_test,predictions))

In [ ]:
# type(predictions)

In [ ]:
real_one = 0
real_zero = 0
pred_one = 0
pred_zero = 0
for i in range(len(predictions)):
    if Y_test[i] == 1:
        real_one += 1
        if predictions[i] == 1:
            pred_one += 1
    else:
        real_zero += 1
        if predictions[i] == 0:
            pred_zero += 1
print("Accuracy for fake reviews in ANN: ",pred_one/real_one)
print("Accuracy for true reviews in ANN: ",pred_zero/real_zero)

In [ ]:
# perm = PermutationImportance(model).fit(X_train, Y_train)
# eli5.show_weights(perm, feature_names = X_train.columns.tolist())

In [ ]:
# svc = SVC(kernel='poly')
# svc.fit(X_train,Y_train)

In [ ]:
# models1 = tf.keras.Sequential([

    
#     layers.Flatten(input_dim = len(X1[0,:])), 
    
#     layers.Dense(units=64, activation=None), 
#     layers.BatchNormalization(),
#     layers.Activation(activations.relu),
    
#     layers.Dense(units=128, activation=None), 
#     layers.BatchNormalization(),
#     layers.Activation(activations.relu),
    
#     layers.Dense(units=128, activation=None), 
#     layers.BatchNormalization(),
#     layers.Activation(activations.relu),
    
#     layers.Dense(units=32, activation=None), 
#     layers.BatchNormalization(),
#     layers.Activation(activations.relu),
    
#     layers.Dense(units=1, activation=activations.sigmoid) 
# ])

# models1.summary()

# models1.compile(
#     optimizer=optimizers.Adam(learning_rate=0.001), 
#     loss=losses.BinaryCrossentropy(), 
#     metrics=['accuracy']
# ) 

In [ ]:
# skf = StratifiedKFold(n_splits=5)
# fold_no = 1
# for train_index, test_index in skf.split(X1, Y):
#     X_train = X1[train_index,:]
#     X_test = X1[test_index,:]
#     Y_train = Y[train_index,:]
#     Y_test = Y[test_index,:]
#     model.fit(X_train, Y_train, epochs=5,batch_size=128,)
#     predictions = model.predict(X_test)
#     print('Fold',str(fold_no),'Accuracy:',accuracy_score(Y_test,np.round(predictions)), ' Precision: ',precision_score(Y_test,np.round(predictions)), ' Recall: ',recall_score(Y_test,np.round(predictions)), ' F1-Score: ',f1_score(Y_test,np.round(predictions)))
#     fold_no += 1

# 2-gram vectorization

In [ ]:
# bigram_vector = CountVectorizer(ngram_range=(2, 2),token_pattern=r'\b\w+\b', min_df=1)
# X_ngram = bigram_vector.fit_transform(data1)

In [ ]:
# svd2 = TruncatedSVD(n_components = 100)
# X_ngram = svd2.fit_transform(X_ngram)

In [ ]:
# X2 = np.hstack((X_ngram,all_features))
# X2.shape

In [ ]:
# models2 = tf.keras.Sequential([

    
#     layers.Flatten(input_dim = len(X1[0,:])), 
    
#     layers.Dense(units=256, activation=None), 
#     layers.BatchNormalization(),
#     layers.Activation(activations.relu),
    
#     layers.Dense(units=128, activation=None), 
#     layers.BatchNormalization(),
#     layers.Activation(activations.relu),
    
#     layers.Dense(units=1, activation=activations.sigmoid) 
# ])

# models2.summary()

# models2.compile(
#     optimizer=optimizers.Adam(learning_rate=0.001), 
#     loss=losses.BinaryCrossentropy(), 
#     metrics=['accuracy']
# ) 

In [ ]:
# skf = StratifiedKFold(n_splits=5)
# fold_no = 1
# for train_index, test_index in skf.split(X2, Y):
#     X_train = X2[train_index,:]
#     X_test = X2[test_index,:]
#     Y_train = Y[train_index,:]
#     Y_test = Y[test_index,:]
#     history1 = models1.fit(X_train, Y_train, epochs=5,batch_size=128,)
#     predictions = models1.predict(X_test)
#     print('Fold',str(fold_no),'Accuracy:',accuracy_score(Y_test,np.round(predictions)), ' Precision: ',precision_score(Y_test,np.round(predictions)), ' Recall: ',recall_score(Y_test,np.round(predictions)), ' F1-Score: ',f1_score(Y_test,np.round(predictions)))
#     fold_no += 1=0.1,
# ) 